In [ ]:
%load_ext autoreload
%autoreload 2

import json

import matplotlib.pyplot as plt
import numpy as np
from omas import load_omas_h5
import xarray as xr
from scipy.interpolate import interp1d

from popsim import PACKAGE_ROOT
from popsim.modules.tearing import ErrorFieldLocking, TearingPhase, ShotPhase, load_overlaps_and_sources
from popsim.simulate import SimInput, simulate, make_time_base

pcs_data = load_omas_h5(f"{PACKAGE_ROOT}/data/tearing/pcs_data_for_onsim2.h5")

overlaps_flattop, coil_sources = load_overlaps_and_sources(f"{PACKAGE_ROOT}/data/tearing/error_field_sources/flattop_01022025.json")
overlaps_startup, _ = load_overlaps_and_sources(f"{PACKAGE_ROOT}/data/tearing/error_field_sources/startup_01022025.json")

metrology = {}
for key1 in overlaps_flattop.keys():
    metrology[key1] = {}
    for key2 in overlaps_flattop[key1].keys():
        if key2 in ['tilt', 'shift']:
            metrology[key1][key2] = 1/np.sqrt(2) *( np.random.random() + 1.j* np.random.random()) # random 0-1 mm, random phase tilt/shift for all coils
        else:
            metrology[key1][key2] = 1.

modes = [(2, 1)]
error_field_locking_config = ErrorFieldLocking.Config(
    modes = modes,
    metrology=metrology,
    overlaps_flattop=overlaps_flattop,
    overlaps_startup=overlaps_startup,
    static_sources={},
    coil_sources=coil_sources,
    hysteresis=0.9,
    efc_efficiency=0.5,
)

error_field_locking_initial_state = ErrorFieldLocking.State(
    W={mode: 0.0 for mode in modes}, 
    F={mode: 0.0 for mode in modes}, 
    mode_phase={mode: 0.0 for mode in modes},
    tearing_phase={mode: TearingPhase.NONE for mode in modes},
)

scaling_laws = json.load(open(f"{PACKAGE_ROOT}/data/tearing/scalinglaws.json"))
scaling_law_terms = scaling_laws["O,L:WLS"]

popsim_time_base = make_time_base(t0=0.0, t1=20, dt=0.1)

pcs_time_base = pcs_data['summary.time']

ods_scaling_law_params = {
    "coeff": 10.,
    "Bt": 12.2,
    "R": 1.85,
    "ne": pcs_data['summary.line_average.n_e.value']/1e19,
    "beta_N": pcs_data['summary.global_quantities.beta_tor_norm.value'],
    "Ip": pcs_data['summary.global_quantities.ip.value'],
    "li": pcs_data['summary.global_quantities.li.value'],
}

ods_pf_active_circuit_current = {}
for i in range(len(pcs_data['pf_active.circuit'])):
    active_circuit_name = pcs_data[f'pf_active.circuit[{i}]']['name']
    if active_circuit_name not in overlaps_flattop.keys():
        print(f"Warning! No overlap data for {active_circuit_name}, skipping")
    else:
        ods_pf_active_circuit_current[active_circuit_name] = pcs_data[f'pf_active.circuit[{i}]']['current']['data']

# For each value in the ods time base, resample the ods time trace to the popsim time base
scaling_law_params = {}
for key, value in ods_scaling_law_params.items():
    if np.isscalar(value):
        scaling_law_params[key] = value
    else:
        interp_function = interp1d(pcs_time_base, value, kind="previous")
        scaling_law_params_array = interp_function(popsim_time_base)
        scaling_law_params[key] = {time: value for time, value in zip(popsim_time_base, scaling_law_params_array)}

pf_active_circuit_current = {}
for key, value in ods_pf_active_circuit_current.items():
    interp_function = interp1d(pcs_time_base, value, kind="previous")
    pf_active_circuit_current_array = interp_function(popsim_time_base)
    pf_active_circuit_current[key] = {time: value for time, value in zip(popsim_time_base, pf_active_circuit_current_array)}

# Say we are in flattop if density is above 70% of the maximum value
shot_phase = {time: (ShotPhase.FLATTOP if value > 0.7 * max(scaling_law_params["ne"].values()) else ShotPhase.STARTUP) for time, value in scaling_law_params["ne"].items()}

error_field_locking_params = ErrorFieldLocking.Params(
    scaling_law_terms=scaling_law_terms,
    scaling_law_params=scaling_law_params,
    pf_active_circuit_current=pf_active_circuit_current,
    shot_phase=shot_phase,
)

error_field_locking_module = ErrorFieldLocking(config=error_field_locking_config)

sim_input = SimInput(time=popsim_time_base, initial_state=error_field_locking_initial_state, params=error_field_locking_params)

sim_xarray = simulate(module=error_field_locking_module, sim_inputs=sim_input)


In [ ]:
import holoviews as hv

hv.extension("matplotlib")

#print(sim_xarray)

def error_field_locking_plots(sim_xarray):
    parameter_plots = []
    for parameter in ["ne", "Ip", "beta_N"]:
        parameter_plots.append(hv.Scatter((sim_xarray.time, sim_xarray[f"params.scaling_law_params.{parameter}"])))
        parameter_plots[-1].opts(ylabel=parameter, aspect=4)

    ef_overlap_plot = hv.Scatter((sim_xarray.time, sim_xarray["output.error_field_overlap"]), label="EF Overlap").opts(aspect=4)
    threshold_plot = hv.Scatter((sim_xarray.time, sim_xarray["output.locking_threshold"]), label="Threshold").opts(aspect=4)
    locked_plot = hv.Scatter((sim_xarray.time, sim_xarray["output.state_dot.tearing_phase.(2, 1)"]), label="Locked").opts(aspect=4)

    module_status_plot = ef_overlap_plot * locked_plot

    all_plots = parameter_plots + [module_status_plot] + [threshold_plot]

    return all_plots

all_plots = error_field_locking_plots(sim_xarray)
# Make the plot very wide
layout = hv.Layout(all_plots).opts(shared_axes=False).cols(1)
layout